In [ ]:
!pip install pymupdf sentence-transformers chromadb

**Build Vector Database**
Purpose: Take your PDF and store its text + embeddings in ChromaDB.
At the end we have:
ChromaDB
 ├── PDF text
 ├── Embeddings
 └── IDs

In [1]:
import fitz #PyMuPdf
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

#step1: Extract text from pdf
def extract_pdf_text(pdf_path):
    doc=fitz.open(pdf_path)
    texts=[]
    for i,page in enumerate(doc):
        text=page.get_text().strip()
        if text:
            texts.append(text)
        else:
            print(f"Skipping empty page{i}")
    return texts

#step 2: initialize embedding model
model=SentenceTransformer("all-MiniLM-L6-v2")

#step 3: initialize chromadb
client=chromadb.Client(Settings())
collection=client.get_or_create_collection(name="pdf_vectors")

#step 4: load pdf and add data
pdf_path="Ratha_Yatra.pdf" #replace with pdf file path
pdf_texts=extract_pdf_text(pdf_path)

documents=[]
embeddings=[]
ids=[]

for i,chunk in enumerate(pdf_texts):
    if chunk.strip(): #ignore empty strings
        print(f"Adding chunk{i}")
        documents.append(chunk)
        vector=model.encode(chunk).tolist()
        embeddings.append(vector)
        ids.append(f"pdf_doc_{i}")

if documents:
    collection.add(documents=documents,embeddings=embeddings,ids=ids)
    print(f"\nSucessfully added{len(documents)} chunks to Vector DB.")
else:
    print("No valid text chunks found in PDF")


#step 5: retrive and display stored data
results=collection.get(include=["documents","embeddings"])

if not results["documents"]:
    print("No documents found in the collection.Check if .add() was successfull." )
else:
    for doc,vec in zip(results["documents"],results["embeddings"]):
        print(f"\nDocument(Preview):{doc[:200]}...")
        print(f"\nVector Dim:{len(vec)},Sample:{vec[:5]}")

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1933.65it/s]


Adding chunk0

Sucessfully added1 chunks to Vector DB.

Document(Preview):Ratha Yatra
The Festival of Chariots
A Journey of the Gods
Ratha Yatra, literally the "chariot journey," is one of the oldest and most spectacular festivals in Hindu tradition.
Once a year, temple dei...

Vector Dim:384,Sample:[-0.00090548  0.01623538 -0.079664   -0.02351834 -0.09068373]


**Retrieve Relevant Information**
Purpose: Test whether ChromaDB can find the relevant PDF content for a question.
Question
"Why do the deities come out of the temple once a year?"
      ↓
Convert question to embedding
      ↓
Search ChromaDB
      ↓
Retrieve relevant PDF text
      ↓
Print the result

In [ ]:
from sentence_transformers import SentenceTransformer
#load model 
model=SentenceTransformer("all-MiniLM-L6-v2")#model name-"all-MiniLM-L6-v2"
#your search question
query_text="Why do the deities come out of the temple once a year?"
query_embedding=model.encode(query_text).tolist()
#query chromadb
results=collection.query(query_embeddings=[query_embedding],n_results=3)#top 3 relevant results

#display results
print("\nTop matches:")
for doc in results["documents"][0]:
    print(f"\n{doc[:300]}...")

**Generate the final Answer**

Retrieved PDF chunks
        +
User question
        ↓
      Prompt
        ↓
      Gemini
        ↓
   Final Answer

   Purpose: Generate a natural-language answer using the retrieved PDF information.

In [ ]:
from dotenv import load_dotenv
from google import genai
import os

# Load variables from .env
load_dotenv()

# Get Gemini API key
api_key = os.getenv("GEMINI_API_KEY")

# Create Gemini client
client = genai.Client(api_key=api_key)


# Step 2: Get the retrieved documents from ChromaDB
retrieved_docs = results["documents"][0]

# Combine the retrieved documents into one context
context = "\n\n".join(retrieved_docs)
# Take all the PDF chunks retrieved from ChromaDB and combine them into one single piece of text, with two new lines between each chunk.


# Step 3: Create prompt for Gemini
prompt = f"""
You are answering a question using information from a PDF.

Use ONLY the context provided below to answer the question.

Context:
{context}

Question:
{query_text}

If the answer is not present in the context,
say: "The answer is not available in the PDF."
"""


# Step 4: Ask Gemini to generate the answer
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)


# Step 5: Display the final answer
print("\nFinal Answer:")
print(response.text)